# Concept Bottleneck Models

In the "[The tale of the deep learning model that failed my driving exam](chapter01_teaser.ipynb)", we compared the decisions of an 18-year-old driver and a deep learning (DL) model when facing a tricky intersection where the driver had a green light, but an ambulance unexpectedly crossed. While both drivers stopped at the intersection, the teenager could explain how she arrived at that conclusion in terms of ***concepts*** like the light color and the presence (or absence) of the ambulance:
>  "I would have crossed if the ambulance wasn't there, but I know for sure that I would never cross in the presence of an ambulance." [Example of a human driver explanation]

In contrast, the DL model, despite making the correct decision and having been trained on millions of instances where lights and ambulances were around, lacked a satisfying decision-making process for the driving evaluator, as the model's explanations were based on raw pixel activations:
>  "If pixel 2890 had an RGB value of (28, 178, 111), the model would have chosen to cross." [Example of a DL driver explanation]

This example highlights the need for developing DL models with a transparent decision-making process like the teenager's, especially in high-stakes fields such as medicine (deciding whether to give a certain treatment), finance (deciding whether to approve a loan), and law (checking whether a hiring system is fair).

## Blueprint for concept-based deep learning models

> ⚠️ **Warning:** The following paragraphs assume a basic understanding of probability.

From a technical standpoint, the problem can be described as follows: we aim to model a relationship between a set of random variables $X \in \mathcal{X}$ representing low-level perceptive features (such as pixels) and a set of random variables corresponding to decisions $Y \in \mathcal{Y}$ (whether to cross or stop).

The DL model in the tale modeled the relationship as $p(Y = \text{cross} \mid X = \text{image’s pixels})$, directly mapping raw image data to a decision.

```{figure} ../images/dnn.png
---
width: 50%
align: center
name: directive-fig
---
A DL model directly maps raw image features (pixels) to a decision (e.g., whether to cross). Image created by the author with assistance from GPT4-o.
```

The teenager, instead, modeled the same problem using intermediate **higher-level variables — referred to as “concepts”, like the light color and the ambulance — leading to a more human-interpretable decision-making process**. Based on such concepts, her reasoning could be expressed as $p(Y = \text{cross} \mid C_1 = \text{light color}, C_2 = \text{ambulance})$, where the concepts $C \in \mathcal{C}$ provided insight into **how** she made a particular decision.

```{figure} ../images/cym.png
---
width: 50%
align: center
name: directive-fig
---
A human driver bases their decisions (e.g., whether to cross) on concepts (e.g., ambulance crossing). Image created by the author.
```

To emulate the teenager's approach, we re-write the probability distribution $p(Y \mid X)$ as if it was obtained by marginalizing the joint distribution $p(Y,C \mid X)$ over the concepts $C$. Now, assuming that $Y$ is conditional independent from $X$ given $C$ we can obtain the following factorization:

$$p(Y \mid X) = \sum_C p(Y, C \mid X) = \sum_C p(Y \mid C) \cdot p(C \mid X)$$

```{figure} ../images/cbm.png
---
width: 50%
align: center
name: directive-fig
---
A concept bottleneck model maps raw image features (pixels) to human-understandable concepts (e.g., ambulance crossing) and then relies on these predicted concepts to make decisions (e.g., whether to cross). Image created by the author with assistance from GPT4-o.
```

In this formulation, the model observes the input features $X$ (such as pixels from an image) and maps them to a set of interpretable, high-level variables $C$, known as "[***concepts***](https://arxiv.org/abs/1711.11279)" {cite}`kim2018interpretability`. These concepts are analogous to the reasoning elements identified by the teenager — such as the traffic light color or the presence of an ambulance. The second part of the model then uses these concepts to determine the decision $Y$ (whether to cross or stop). This class of models is known as a [**Concept Bottleneck Model**](https://arxiv.org/abs/2007.04612) (CBM) {cite}`koh2020concept`. CBMs parametrize the conditional distributions with a pair of neural networks:
- A "*concept encoder*" $g$  that takes as an input a sample $x \in \mathcal{X}$, and predicts concepts $c \in \mathcal{C}$. This network parametrizes the concept distribution.
- A "*task predictor*" $f$ that takes as an input a concept tuple $c \in \mathcal{C}$ and predicts an output label $y \in \mathcal{Y}$. This network parametrizes the output distribution.

Given a concept-based dataset of i.i.d. triples (input, concepts, task) $D = \{(\hat{x}, \hat{c}, \hat{y})\}$, the CBM’s parameters ($\theta_g$ and $\theta_f$) are usually optimized via gradient descent by maximizing the log-likelihood:

$$\max_{\theta_g, \theta_f} \mathcal{L}(\theta_g, \theta_f, D) = \sum_{(\hat{x}, \hat{c}, \hat{y}) \in D} \log p(Y=\hat{y}, C=\hat{c} \mid X=\hat{x}; \theta_g, \theta_f) =\\
= \sum_{(\hat{x}, \hat{c}, \hat{y}) \in D} \log p(Y=\hat{y} \mid C=\hat{c}; \theta_f) + \log p(C=\hat{c} \mid X=\hat{x}; \theta_g)$$

In CBMs there are three different ways to optimize this objective function:
1. **Independent training** optimizes $f$ and $g$ independently: $f$ is trained using ground truth concepts $\hat{c}$ as input. At test time, $f$ takes $g(\hat{x})$ as input.
2. **Sequential training** first optimizes $g$ independently. Once trained, the parameters of the concept encoder $g$ are frozen, and $f$ is trained using $g(\hat{x})$ as input.
3. **Joint training**: optimizes $f$ and $g$ at the same time and $f$ takes as input $g(\hat{x})$.

The following coding practice introduces you to implementing CBMs and shows how to query CBMs to understand the model's decision-making process.


## Coding practice

In this practice, we implement a Concept Bottleneck Model (CBM) for a simple traffic light scenario where decisions to cross or stop depend on two concepts: the traffic light being green and the presence of an ambulance. The model predicts these concepts and uses them to make decisions.

> ⚠️ **Warning:** This section assumes a basic understanding of programming machine learning scripts using deep learning frameworks, specifically PyTorch. If you're new to PyTorch, consider starting with [these introductory tutorials](https://pytorch.org/tutorials/beginner/basics/intro.html) to get up to speed.


### Step #1: Install PyC

First, we install the necessary Python packages required to implement CBMs. This includes our library `pytorch-concepts` which runs on top of standard deep learning libraries (`PyTorch` and `torch_geometric`).

In [1]:
%%capture
!pip install torch
!pip install torch_geometric
!pip install -i https://test.pypi.org/simple/ --upgrade pytorch-concepts

### Step #2: Load traffic scenario

Next, we load the dataset for our traffic light scenario. This dataset comprises a collection of images of various road intersections under different conditions. Each image is annotated with the following class labels:
*   **Traffic Light Color**: Indicates the current color of the traffic light (e.g., red, yellow, green).
*   **Presence of an Ambulance**: Specifies whether an ambulance is visible in the scene.
*   **Decision to Cross**: Denotes whether the appropriate action is to cross the intersection or to stop.

In [2]:
from torch_concepts.data import TrafficLights

n_samples = 1000

# Loading dataset
dataset = TrafficLights(n_samples=n_samples)
x_train, c_train, y_train, concept_names, task_names = dataset.x_train, dataset.c_train, dataset.y_train, dataset.concept_names, dataset.task_names

# Example of scenario
print(x_train.shape)
print(c_train.shape, c_train[0], concept_names)
print(y_train.shape, y_train[0], task_names)

torch.Size([1000, 10])
torch.Size([1000, 2]) tensor([0., 1.]) ['traffic light green', 'ambulance crossing']
torch.Size([1000, 1]) tensor([0.]) ['cross']


### Step #3: Define the CBM

We are now ready to construct our first CBM. The model comprises three main components: encoder, concept predictor, and task predictor.
These components are combined sequentially to form the complete CBM architecture.

**Encoder**: The encoder takes as input low-level features $X$ and produces a lower-dimensional representation. The specific architecture of the encoder depends on the input data type; it could be a convolutional neural network, a recurrent neural network, a transformer, or another suitable model.

In [3]:
import torch

latent_dims = 5

# The encoder extracts a low-dimensional representation of the input
encoder = torch.nn.Sequential(
    torch.nn.Linear(x_train.shape[1], latent_dims),
    torch.nn.LeakyReLU()
)
encoder

Sequential(
  (0): Linear(in_features=10, out_features=5, bias=True)
  (1): LeakyReLU(negative_slope=0.01)
)

**Concept Layer**: A concept layer takes an embedding as input and generates a set of concept representations, which in this case correspond to concept logits. To implement concept layers, we use the [`PyC`](https://github.com/pyc-team/pytorch_concepts) library.

To define this layer, the following inputs are required:
- `in_features`: The size of the embedding generated by the encoder.
- `out_concept_dimensions`: A dictionary where each key represents a dimension of the concept tensor, and each value is a list of strings assigning concept names to the elements of that dimension.

In our example, this layer produces two concept logits corresponding to "traffic light green" and "ambulance crossing".

In [4]:
from torch_concepts.nn import ConceptLayer

# The concept scorer predicts concept logits {traffic light color, ambulance presence}
c_layer = ConceptLayer(
    in_features=latent_dims,
    out_concept_dimensions={1: concept_names}
)
print('Architecture:\n', c_layer)
print('Expected concepts in output tensor:\n', c_layer.concept_names)

ImportError: cannot import name 'ConceptLayer' from 'torch_concepts.nn' (/mnt/3106DB277DCCEAA5/anaconda3/envs/nips/lib/python3.8/site-packages/torch_concepts/nn/__init__.py)

In `PyC` concept layers produce as output an object of type `ConceptTensor` whose dimensions are labeled with the provided concept names:

In [ ]:
concept_tensor = c_layer(torch.randn(1, latent_dims))
print(concept_tensor)

The object `ConceptTensor` has built-in methods to extract concept representations from specific dimensions which might be practical for in-depth analysis:

In [ ]:
concept_traffic_light = concept_tensor.extract_by_concept_names({1: ['traffic light green']})
print(concept_traffic_light)

**Task Predictor**: Takes as input the predicted concept representations to determine the final decision on whether to cross or not. In this case, we implement the task predictor with a multi-layer perceptron, but we will discuss more advanced (and interpretable) task predictors later on in this series.


In [ ]:
# The task predictor determines the value of the downstream label {cross}
y_predictor = torch.nn.Sequential(
    torch.nn.Linear(c_train.shape[1], latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, y_train.shape[1])
)
y_predictor

### Step #4: Train the CBM

We set the training parameters, including the number of epochs (`n_epochs`) and learning rate (`lr`). Using the Adam optimizer and Binary Cross-Entropy Loss, we train the CBM through a standard PyTorch loop. In each epoch, the model performs a forward pass to predict concepts and the final decision, computes the combined loss, performs backpropagation, and updates the model parameters. The concept regularization weight (`concept_reg`) controls the weight of the concept loss w.r.t. the downstream task loss. In this example we use a joint training as we optimize all models at the same time and we pass as input to the task predictor the concept activations produced by the concept layer.

In [4]:
n_epochs = 1000
concept_reg = 0.5
lr = 0.01

# Define optimizer and loss function
model = torch.nn.Sequential(encoder, c_layer, y_predictor)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
loss_fn = torch.nn.BCELoss()

# Standard PyTorch learning cycle
model.train()
for epoch in range(n_epochs):
   optimizer.zero_grad()

   # Encode input, then predict concept and downstream tasks activations
   emb = encoder(x_train)
   c_pred = c_layer(emb).sigmoid()
   y_pred = y_predictor(c_pred).sigmoid()

   # Double loss on concepts and tasks
   loss = concept_reg * loss_fn(c_pred, c_train) + loss_fn(y_pred, y_train)
   loss.backward()
   optimizer.step()

   if epoch % 100 == 0:
       print(f"Epoch {epoch}: Loss {loss.item():.2f}")

Epoch 0: Loss 1.07
Epoch 100: Loss 0.51
Epoch 200: Loss 0.32
Epoch 300: Loss 0.14
Epoch 400: Loss 0.08
Epoch 500: Loss 0.06
Epoch 600: Loss 0.05
Epoch 700: Loss 0.04
Epoch 800: Loss 0.03
Epoch 900: Loss 0.03


### Step #5: Trace task prediction back to concept activations

We examine how the concept activations (green light and ambulance presence) influence the downstream task prediction. By testing the model with a sample input, we observe the resulting decision.


In [5]:
model.eval()
print(f"Task ({task_names}): {y_pred[0]>0.5}")
print(f"Concepts ({concept_names}): {c_pred[0]>0.5}")

Task (['cross']): tensor([False])
Concepts (['traffic light green', 'ambulance crossing']): tensor([False,  True])


The model correctly identifies that the traffic light is green and there is no ambulance, and it decides to cross (task prediction is `True`).

### Step #6: Change concept activations to affect task predictions

Finally, we alter the concept activations to different values and observe how these changes influence the downstream task prediction. This demonstrates how the model's decisions are directly tied to specific concept activations.


In [6]:
# Intervene changing the value of the concept "ambulance" to True
c_intervened = c_pred[0].clone().unsqueeze(0)
c_intervened[0, 1] = 1

# Compute new task prediction
y_intervened = y_predictor(c_intervened).sigmoid()

print(f"Concepts: {c_intervened[0]>0.5}")
print(f"Task: {y_intervened[0]>0.5}")

Concepts: tensor([False,  True])
Task: tensor([False])


The model effectively reacts to the concept intervention<sup>1</sup> predicting that in the presence of a green light and of an ambulance, the car should stop (task prediction is `False`).

## Take home message
In summary, in this chapter we demonstrated how to implement Concept Bottleneck Models (CBMs). These models are inherently explainable as:
*    **Task predictions can be traced back to the activation of human-interpretable concepts**. This enables the model to answer the driving evaluator's question by saying, "I decided to cross as I saw a green light and there was no ambulance".
*    **Altering concept values changes the model's decisions**. This allows the model to respond to the evaluator's question with, "In the same scenario, if there is an ambulance, I would cross".


The [next chapter](chapter03_metrics.ipynb) will discuss key metrics used in concept-based interpretability to quantitatively evaluate CBM performance.

**Bibliography**


Koh, Pang Wei, et al. "Concept bottleneck models." International conference on machine learning. PMLR, 2020.

Kim, Been, et al. "Interpretability beyond feature attribution: Quantitative testing with concept activation vectors (tcav)." International conference on machine learning. PMLR, 2018.



--------------------
<sup>1</sup>: The ability of CBMs in responding to concept interventions can be used to improve the model's performance by making human experts fix mispredicted concepts. This topic will be discussed in detail in Chapter 3.